In [1]:
# CELL 1: Install Required Libraries
# We force upgrade sympy and torchao to resolve recent Colab version conflicts
!pip install -q --upgrade sympy torchao
!pip install -q transformers peft pandas Pillow torchvision tqdm
print("✅ Libraries installed successfully!")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\Charlene C. Dilig\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


✅ Libraries installed successfully!



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\Charlene C. Dilig\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# CELL 2: Mount Drive and Define Transforms
import torchvision.transforms as T
import torch

# 2. Define the Augmentation for TRAINING (V2.1 Precision Jitter)
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    # --- V2.1 TWEAK: Mid-Range Jitter (0.4) for better precision ---
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    # --------------------------------------------------------------
    T.ToTensor(),
    T.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    T.ToPILImage()
])

# 3. Define the rule for VALIDATION
eval_transform = T.Compose([
    T.Resize((224, 224))
])

print("✅ Cell 2 Updated: Precision Augmentations defined!")

✅ Cell 2 Updated: Precision Augmentations defined!


In [3]:
# CELL 3: The PyTorch Dataset Class & HARD NEGATIVE MINING SAMPLER
import pandas as pd
import os
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

class LostAndFoundDataset(Dataset):
    def __init__(self, csv_file, image_dir, split_type, transform=None):
        full_data = pd.read_csv(csv_file)
        self.data = full_data[full_data['split'] == split_type].reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        lost_img_path = os.path.join(self.image_dir, row['lost_image'])
        lost_image = Image.open(lost_img_path).convert('RGB')

        if self.transform:
            lost_image = self.transform(lost_image)

        return {
            'lost_image': lost_image,
            'positive_caption': row['positive_caption'],
            'hard_negative_caption': row['hard_negative_caption']
        }

def custom_collate(batch):
    return {
        'lost_image': [item['lost_image'] for item in batch],
        'positive_caption': [item['positive_caption'] for item in batch],
        'hard_negative_caption': [item['hard_negative_caption'] for item in batch]
    }

# ==========================================
# 🚨 NEW: THE HARD NEGATIVE MINING SAMPLER 🚨
# ==========================================
class CategoryBatchSampler(Sampler):
    def __init__(self, dataset_df, batch_size):
        self.batch_size = batch_size

        # Group all item indices by their category (the word before the '_')
        self.category_to_indices = defaultdict(list)
        for idx, row in dataset_df.iterrows():
            filename = row['lost_image']
            category = filename.split('_')[0]
            self.category_to_indices[category].append(idx)

    def __iter__(self):
        batches = []
        for cat, indices in self.category_to_indices.items():
            random.shuffle(indices)
            for i in range(0, len(indices), self.batch_size):
                chunk = indices[i:i + self.batch_size]
                if len(chunk) > 1:
                    batches.append(chunk)

        random.shuffle(batches)
        for batch in batches:
            yield batch

    def __len__(self):
        return sum((len(indices) // self.batch_size) + (1 if len(indices) % self.batch_size > 1 else 0)
                   for indices in self.category_to_indices.values())

# ==========================================
# INITIALIZATION
# ==========================================
CSV_PATH = '../captions/dataset_captionsV2.csv'
IMG_DIR = '../images'

# Load the Datasets
train_dataset = LostAndFoundDataset(CSV_PATH, IMG_DIR, split_type='train', transform=train_transform)
val_dataset = LostAndFoundDataset(CSV_PATH, IMG_DIR, split_type='val', transform=eval_transform)

# 🚨 THE MAGIC SWITCH: Attach the Hard Negative Sampler 🚨
hn_sampler = CategoryBatchSampler(train_dataset.data, batch_size=8)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=hn_sampler, # <-- Replaces shuffle=True
    collate_fn=custom_collate
)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=custom_collate)

print(f"✅ Loaded {len(train_dataset)} Training Pairs using Category-Level Hard Negative Mining!")
print(f"✅ Loaded {len(val_dataset)} Validation Pairs.")

✅ Loaded 420 Training Pairs using Category-Level Hard Negative Mining!
✅ Loaded 90 Validation Pairs.


In [4]:
# CELL 4: Model Architecture & LoRA
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model

model_id = "qihoo360/fg-clip-base"
processor = CLIPProcessor.from_pretrained(model_id)
base_model = CLIPModel.from_pretrained(model_id)

# --- V2.2 TWEAK: Higher LoRA Rank for More Fine-Grained Capacity ---
lora_config = LoraConfig(
    r=64,                 # <--- INCREASED from 32
    lora_alpha=128,       # <--- Kept at 128
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none"
)

lora_model = get_peft_model(base_model, lora_config)
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_model.to(device)

print("✅ Cell 4 Updated: LoRA rank increased to r=64 for finer-grained alignment!")
lora_model.print_trainable_parameters()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Cell 4 Updated: LoRA rank increased to r=64 for finer-grained alignment!
trainable params: 7,864,320 || all params: 157,485,057 || trainable%: 4.9937


In [5]:
print("============================================================")
print("🔎 PRE-TRAINING MODEL ARCHITECTURE INSPECTION")
print("============================================================")

try:
    # We reach directly into the PyTorch architecture of the loaded base_model
    # to find the convolutional layer responsible for chopping the image.
    patch_layer = base_model.vision_model.embeddings.patch_embedding

    patch_size = patch_layer.kernel_size[0]
    image_size = 224 # Standard CLIP input size
    num_patches = (image_size // patch_size) ** 2

    print(f"Target Layer: {patch_layer}")
    print(f"Kernel Size:  {patch_layer.kernel_size}  <-- This is your Patch Size!")
    print(f"Stride:       {patch_layer.stride}")

    print("\n--- Physical Grid Calculation ---")
    print(f"Math: {image_size} / {patch_size} = {image_size // patch_size}")
    print(f"Grid: {image_size // patch_size} x {image_size // patch_size} = {num_patches} total patches.")
    print(f"Tensor Shape: [1, {num_patches + 1}, 768] (Includes +1 for the CLS Token)")

    print("\n============================================================")
    print("🎯 FINAL VERDICT")
    print("============================================================")
    if patch_size == 16:
        print("✅ CONFIRMED: You are officially fine-tuning a ViT-B/16 model.")
    elif patch_size == 32:
        print("❌ WARNING: This is a ViT-B/32 model. Update your methodology!")
    else:
        print(f"⚠️ UNKNOWN: Patch size is {patch_size}.")

except AttributeError:
    print("⚠️ Error: Could not locate the patch_embedding layer. Ensure your variable is named 'base_model'.")

🔎 PRE-TRAINING MODEL ARCHITECTURE INSPECTION
Target Layer: Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
Kernel Size:  (16, 16)  <-- This is your Patch Size!
Stride:       (16, 16)

--- Physical Grid Calculation ---
Math: 224 / 16 = 14
Grid: 14 x 14 = 196 total patches.
Tensor Shape: [1, 197, 768] (Includes +1 for the CLS Token)

🎯 FINAL VERDICT
✅ CONFIRMED: You are officially fine-tuning a ViT-B/16 model.


In [6]:
# CELL 5: The V2.2 Training Loop — Lower Temp, Lower LR, More Capacity, Gradient Accumulation
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup

# --- V2.2 TWEAKS: Sharper Temp + Slower LR + Gradient Accumulation ---
ACCUMULATION_STEPS = 4          # <--- NEW: effective batch = 8 * 4 = 32
EFFECTIVE_BATCH = 8 * ACCUMULATION_STEPS
optimizer = AdamW(filter(lambda p: p.requires_grad, lora_model.parameters()), lr=1e-5)  # <--- LOWERED from 3e-5
STRICT_TEMP = 0.01             # <--- LOWERED from 0.03 (sharper similarity distribution)
# ---------------------------------------------------------------------

loss_fn = nn.CrossEntropyLoss()
num_epochs = 30                # <--- EXTENDED from 20
total_steps = (len(train_loader) * num_epochs) // ACCUMULATION_STEPS
warmup_steps = int(0.10 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val_loss = float('inf')
patience = 8                   # <--- INCREASED from 5
patience_counter = 0
save_path = "../lorafinetuned/fgclip-lora-finetunedV2.2-1200-HNM"  # <--- FIXED to V2.2

for epoch in range(num_epochs):
    lora_model.train()
    total_train_loss = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    optimizer.zero_grad()

    for step, batch in enumerate(train_bar):
        images = batch['lost_image']
        all_texts = batch['positive_caption'] + batch['hard_negative_caption']
        inputs = processor(text=all_texts, images=images, return_tensors="pt", padding=True).to(device)
        outputs = lora_model(**inputs)

        image_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)

        custom_logits = (image_embeds @ text_embeds.T) / STRICT_TEMP
        labels = torch.arange(len(images)).to(device)
        loss = loss_fn(custom_logits, labels)
        loss = loss / ACCUMULATION_STEPS  # <--- Scale loss by accumulation steps

        loss.backward()

        if (step + 1) % ACCUMULATION_STEPS == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_train_loss += loss.item() * ACCUMULATION_STEPS

    # Validation
    lora_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['lost_image']
            inputs = processor(text=batch['positive_caption'] + batch['hard_negative_caption'],
                             images=images, return_tensors="pt", padding=True).to(device)
            outputs = lora_model(**inputs)
            i_emb = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
            t_emb = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
            logits = (i_emb @ t_emb.T) / STRICT_TEMP
            total_val_loss += loss_fn(logits, torch.arange(len(images)).to(device)).item()
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"\n=> Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        lora_model.save_pretrained(save_path)
        print(f"🟢 Improvement! Val loss: {avg_val_loss:.4f} | Saved to {save_path}\n")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("🛑 EARLY STOPPING.")
            break

print(f"\n✅ Training complete. Best val loss: {best_val_loss:.4f}")

import os
import datetime
print("\n--- Verifying Saved Files ---")

for f in os.listdir(save_path):
    full = os.path.join(save_path, f)
    mod_time = os.path.getmtime(full)
    readable_time = datetime.datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M:%S')
    print(f"📄 {f}: Saved at {readable_time}")

from peft import PeftModel
from transformers import CLIPModel
import json

print("\n--- Verifying Adapter Weights ---")
base = CLIPModel.from_pretrained("qihoo360/fg-clip-base")
loaded = PeftModel.from_pretrained(base, save_path)

with open(os.path.join(save_path, "adapter_config.json"), "r") as f:
    config = json.load(f)
print("✅ Adapter Config Loaded Successfully:")

print(json.dumps(config, indent=2))

Epoch 1/30 [Train]: 100%|██████████| 53/53 [00:46<00:00,  1.13it/s]



=> Epoch 1 Summary | Train Loss: 0.9966 | Val Loss: 1.7328
🟢 Improvement! Val loss: 1.7328 | Saved to ../lorafinetuned/fgclip-lora-finetunedV2.2-1200-HNM



Epoch 2/30 [Train]: 100%|██████████| 53/53 [00:48<00:00,  1.10it/s]



=> Epoch 2 Summary | Train Loss: 1.0377 | Val Loss: 1.7238
🟢 Improvement! Val loss: 1.7238 | Saved to ../lorafinetuned/fgclip-lora-finetunedV2.2-1200-HNM



Epoch 3/30 [Train]: 100%|██████████| 53/53 [00:45<00:00,  1.16it/s]


...


=> Epoch 28 Summary | Train Loss: 0.6089 | Val Loss: 1.5614
🟢 Improvement! Val loss: 1.5614 | Saved to ../lorafinetuned/fgclip-lora-finetunedV2.2-1200-HNM



Epoch 29/30 [Train]: 100%|██████████| 53/53 [00:43<00:00,  1.23it/s]



=> Epoch 29 Summary | Train Loss: 0.5563 | Val Loss: 1.5612
🟢 Improvement! Val loss: 1.5612 | Saved to ../lorafinetuned/fgclip-lora-finetunedV2.2-1200-HNM



Epoch 30/30 [Train]: 100%|██████████| 53/53 [00:44<00:00,  1.20it/s]



=> Epoch 30 Summary | Train Loss: 0.5048 | Val Loss: 1.5612
🟢 Improvement! Val loss: 1.5612 | Saved to ../lorafinetuned/fgclip-lora-finetunedV2.2-1200-HNM


✅ Training complete. Best val loss: 1.5612

--- Verifying Saved Files ---
📄 adapter_config.json: Saved at 2026-05-04 22:05:16
📄 adapter_model.safetensors: Saved at 2026-05-04 22:05:16
📄 README.md: Saved at 2026-05-04 22:05:16

--- Verifying Adapter Weights ---
✅ Adapter Config Loaded Successfully:
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": {
    "base_model_class": "CLIPModel",
    "parent_library": "transformers.models.clip.modeling_clip"
  },
  "base_model_name_or_path": "qihoo360/fg-clip-base",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transfo

In [7]:
# 📊 CELL: The Exhaustive "Final Boss" Experiment Summary
def print_final_boss_summary():
    import torch, transformers, peft, platform
    
    out = []
    out.append("=" * 80)
    out.append("🔬 FINAL BOSS: EXHAUSTIVE EXPERIMENT CONFIGURATION & RESULTS SUMMARY")
    out.append("=" * 80)
    
    # 1. Hardware & Environment
    out.append("\n[1] HARDWARE & ENVIRONMENT")
    try: 
        out.append(f"    🖥️  OS:              {platform.system()} {platform.release()}")
        out.append(f"    🐍 Python Version:  {platform.python_version()}")
        out.append(f"    📦 PyTorch Version: {torch.__version__}")
        out.append(f"    📦 Transformers:    {transformers.__version__}")
        out.append(f"    📦 PEFT Version:    {peft.__version__}")
        if torch.cuda.is_available():
            out.append(f"    🚀 GPU:             {torch.cuda.get_device_name(0)}")
            mem_used = torch.cuda.max_memory_allocated() / (1024 ** 3)
            out.append(f"    💾 Max VRAM Used:   {mem_used:.2f} GB")
        else:
            out.append("    🐌 GPU:             None (CPU Training)")
        out.append(f"    🎲 Global Seed:     {torch.initial_seed()}")
    except Exception as e: out.append(f"    ⚠️  Error: {e}")

    # 2. Base Model & Dataset
    out.append("\n[2] MODEL & DATASET")
    try: out.append(f"    🤖 Base Model ID:       {model_id}")
    except NameError: pass
    try: out.append(f"    📏 Max Text Length:     {processor.tokenizer.model_max_length} tokens")
    except NameError: pass
    try: out.append(f"    🔤 Text Padding:        True (Longest in batch)")
    except Exception: pass
    try:
        out.append(f"    🖼️  Train Images:        {len(train_dataset)}")
        out.append(f"    🖼️  Validation Images:   {len(val_dataset)}")
    except NameError: pass

    try:
        if 'hn_sampler' in globals():
            out.append(f"    ⛏️  Mining Strategy:     Hard Negative Category Mining")
        else:
            out.append(f"    ⛏️  Mining Strategy:     Standard Random Shuffle")
    except Exception: pass

    # 3. Model Architecture Details
    out.append("\n[3] ARCHITECTURE & LoRA")
    try:
        trainable, total = lora_model.get_nb_trainable_parameters()
        out.append(f"    📊 Trainable Params:    {trainable:,} ({(trainable/total)*100:.4f}% of total)")
        out.append(f"    🧠 LoRA Rank (r):       {lora_config.r}")
        out.append(f"    🧠 LoRA Alpha:          {lora_config.lora_alpha}")
        out.append(f"    🎯 Target Modules:      {lora_config.target_modules}")
        out.append(f"    💧 Dropout:             {lora_config.lora_dropout}")
        out.append(f"    🧮 Model dtype:         {lora_model.dtype}")
    except NameError: pass

    # 4. Hyperparameters & Optimizer
    out.append("\n[4] HYPERPARAMETERS & TRAINING")
    try: 
        out.append(f"    ⚙️  Optimizer:           {type(optimizer).__name__}")
        pg = optimizer.param_groups[0]
        out.append(f"    📉 Start Learn Rate:    {pg['lr']}")
        out.append(f"    📉 End Learn Rate:      {pg['lr']}")
        out.append(f"    ⚓ Weight Decay:        {pg.get('weight_decay', 0.0)}")
        out.append(f"    🎢 Betas (B1, B2):      {pg.get('betas', 'N/A')}")
        out.append(f"    🔢 Optimizer Epsilon:   {pg.get('eps', 'N/A')}")
    except NameError: pass
    try: out.append(f"    ⚖️  Loss Function:       {type(loss_fn).__name__}")
    except NameError: pass
    
    try:
        if 'ACCUMULATION_STEPS' in globals():
            out.append(f"    🔁 Gradient Accum.:     {ACCUMULATION_STEPS} (effective batch = 8 × {ACCUMULATION_STEPS} = {8 * ACCUMULATION_STEPS})")
    except Exception: pass
    
    try: 
        if 'STRICT_TEMP' in globals():
            out.append(f"    🌡️  Temperature:         {STRICT_TEMP} (Manual Strict Override — V2.2: lowered from 0.03)")
        else:
            actual_temp = 1.0 / lora_model.base_model.logit_scale.exp().item()
            out.append(f"    🌡️  Temperature:         {actual_temp:.4f} (Auto-Scaled by HF Logit Parameter)")
    except Exception: pass

    try: 
        out.append(f"    📦 Physical Batch Size: 8 (per sampler)")
        out.append(f"    📦 Effective Batch:     {8 * ACCUMULATION_STEPS} (with gradient accumulation)")
        out.append(f"    📦 Val Batch Size:      {val_loader.batch_size}")
        out.append(f"    👷 Dataloader Workers:  {train_loader.num_workers}")
    except NameError: pass
    
    try: 
        out.append(f"    📈 Total Steps:         {total_steps}")
        if 'warmup_steps' in globals():
            out.append(f"    🔥 Warmup Steps:        {warmup_steps} (Cosine Scheduler)")
    except NameError: pass

    # 5. Training Results
    out.append("\n[5] TRAINING RESULTS")
    try: out.append(f"    📉 Train Loss (at Stop): {avg_train_loss:.4f}")
    except NameError: pass
    try: out.append(f"    📉 Val Loss (at Stop):   {avg_val_loss:.4f}")
    except NameError: pass
    try: out.append(f"    🏆 Best Val Loss:        {best_val_loss:.4f} (Model Saved Here!)")
    except NameError: pass
    try: out.append(f"    🏁 Stopped at Epoch:     {epoch+1} / {num_epochs}")
    except NameError: pass
    try: out.append(f"    🛑 Early Stop Patience:  {patience} epochs")
    except NameError: pass


    # 6. Output Details
    out.append("\n[6] OUTPUT")
    try: out.append(f"    💾 Saved To:            {save_path}")
    except NameError: pass

    # 7. Augmentations
    out.append("\n[7] DATA AUGMENTATIONS")
    try:
        out.append(str(train_transform))
    except NameError:
        out.append("    ⚠️  Could not find train_transform variable.")
        
    out.append("=" * 80)
        # === NEW: PRINT AND SAVE TO FILE ===
    report_text = "\n".join(out)
    print(report_text)

    try:
        import os
        if 'save_path' in globals():
            # Saves it right next to your weights!
            txt_path = os.path.join(save_path, "training_summary_report.txt")
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(report_text)
            print(f"\n💾 Training Summary safely saved to: {txt_path}")
    except Exception as e:
        print(f"\n⚠️ Could not save text file: {e}")

# Run the function
print_final_boss_summary()



🔬 FINAL BOSS: EXHAUSTIVE EXPERIMENT CONFIGURATION & RESULTS SUMMARY

[1] HARDWARE & ENVIRONMENT
    🖥️  OS:              Windows 10
    🐍 Python Version:  3.13.3
    📦 PyTorch Version: 2.6.0+cu124
    📦 Transformers:    4.57.1
    📦 PEFT Version:    0.19.1
    🚀 GPU:             NVIDIA GeForce RTX 3050 Laptop GPU
    💾 Max VRAM Used:   2.29 GB
    🎲 Global Seed:     31433333166800

[2] MODEL & DATASET
    🤖 Base Model ID:       qihoo360/fg-clip-base
    📏 Max Text Length:     77 tokens
    🔤 Text Padding:        True (Longest in batch)
    🖼️  Train Images:        420
    🖼️  Validation Images:   90
    ⛏️  Mining Strategy:     Hard Negative Category Mining

[3] ARCHITECTURE & LoRA
    📊 Trainable Params:    7,864,320 (4.9937% of total)
    🧠 LoRA Rank (r):       64
    🧠 LoRA Alpha:          128
    🎯 Target Modules:      {'q_proj', 'v_proj', 'k_proj', 'out_proj'}
    💧 Dropout:             0.1
    🧮 Model dtype:         torch.float32

[4] HYPERPARAMETERS & TRAINING
    ⚙️  Optimizer: